In [2]:
import WorkFlowANN as flow
import requests
import torch
import numpy as np
import NeuralNetwork as NN
import pickle

Update games!

In [3]:
year = 2024
month = 12
day = 12
update_seasons = 'true'
start_year=year
end_year=year+1

requests.get(f'http://127.0.0.1:5000/update_games/year={year}/month={month}/day={day}/update_seasons={update_seasons}/start_year={start_year}/end_year={end_year}')

<Response [200]>

In [243]:
first_name = 'Devin'
last_name = 'Vassell'
middle_name = None
#Get player id
if middle_name == None:
    apiinfo = requests.get(f'http://127.0.0.1:5000/get_player_id/first_name={first_name}/last_name={last_name}')
else:
    apiinfo = requests.get(f'http://127.0.0.1:5000/get_player_id/first_name={first_name}/last_name={last_name}/middle_name={middle_name}')

player_id = int(apiinfo.json()['id'])

#Check if player has a model created
apiinfo = requests.get(f'http://127.0.0.1:5000/get_player_model/player_id={player_id}')

player_model = apiinfo.json()

data = list(player_model.keys())

#If no model is created, generate it
if 'ERROR' in data:
    model = NN.MovingAverageModel()
    model.train()

    training_features, training_labels, test_features, test_labels = flow.obtain_required_data(first_name, last_name)

    player_model, loss_evolution_nonlinear = flow.train_model(training_features, training_labels, model, epochs=100, learning_rate=0.0001, batch_size=200)
    player_model.eval()

    predictions = player_model(torch.tensor(test_features).float()).detach().numpy()
    random_variable = [item[0] for item in predictions-test_labels]

    params = flow.retrieve_skewness_params(random_variable)
    
    with open('model.pkl', 'wb') as file:
        pickle.dump(model, file)

    files = {'model': open('model.pkl', 'rb')}

    url = f'http://127.0.0.1:5000/input_player_model/player_id={player_id}/skewness={params[0]}/mean={params[1]}/scale={params[2]}'

    input_info = requests.post(url, files=files)

else:
    with open('model.pkl', 'rb') as file:
        params = float(player_model['skewness']), float(player_model['mean']), float(player_model['scale'])
        player_model = pickle.load(file)

In [246]:
opponent = 'Portland Trail Blazers'
point_total = 18.5

player_features = flow.retrieve_player_features(first_name, last_name, middle_name, 2025)
team_features = flow.retrieve_team_features(opponent)

prediction_features = np.concatenate((player_features, team_features))

prediction, probability, bet = flow.calculate_probability(params, player_model, prediction_features, point_total)

print(f'Predicted Points Score: {prediction+point_total}')
print(f'There is a {round(probability * 100,4)} % chance {first_name} {middle_name} {last_name} scores {bet} {point_total}')

Predicted Points Score: 12.5
There is a 72.7349 % chance Anfernee None Simons scores UNDER 18.5


In [234]:
winnings = flow.expected_winnings(1-probability, 154, 1)
print(round(winnings,4))

0.4199


In [ ]:
from scipy.stats import skewnorm
import matplotlib.pyplot as plt
x = np.linspace(-40, 40, 1000)
y = skewnorm.pdf(x, *params)

print(params)

plt.plot(x,y)
plt.show()